IF YOU HAVE ANY PROBLEMS, LET THE TEACHER KNOW!

TODO SUMMARY IN THIS FILE:

1. <a href="#dataleakage">Questions to answer - data leakage</a>
2. <a href="#classimbalance">Complete the code - class imbalance</a>
3. <a href="#overfitting">Questions to answer - overfitting</a>
4. <a href="#overfitting">Complete the code - overfitting</a>
5. <a href="#multiclass">Complete the code - multiclass classification (additional task)</a>

In [ ]:

# HOW TO CREATE CONDA ENV
"""conda create -n dl python=3.10
conda activate dl

pip install torch torchvision medmnist
pip install grad-cam
pip install seaborn"""


# IF YOU USE GOOGLE COLAB

"""!pip install python==3.10
!pip install medmnist
!pip install grad-cam
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126
!
!pip install pillow==11.1.0

from google.colab import drive
drive.mount('/content/drive')

import sys
import os
os.chdir('/content/drive/MyDrive/DL/')"""


# Libraries required: torch, seaborn, medmnist, grad-cam

In [ ]:
import importlib
import src.train_utils
import src.utils
import torch

importlib.reload(src.utils)
importlib.reload(src.train_utils)

# SPRAWDŹ CZY TORCH JEST ZAINSTALOWANY POPRAWNIE 

import torch
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
print("Current device:", torch.cuda.current_device() if torch.cuda.is_available() else "CPU")

## 2. SELECTED PROBLEMS THAT MAY APPEAR DURING CNN TRAINING

### 2.1 Data leakage

The data used in this task came from over 1,200 patients. Some patients had multiple examinations performed (so several images could have been from the same patient). The database was randomly divided into training, validation, and test sets – patients could therefore appear in different subsets simultaneously.

Additionally, a separate test set was prepared from a different source.

``Examine the code snippet below. Run it without making any changes and ANSWER THE QUESTIONS regarding the data leak.``

In [ ]:
from src.utils import data_leakage_dataset, classification_training, classification_validate, binary_classification_GRADCAM
from torch.utils.data import DataLoader, Subset, ConcatDataset, WeightedRandomSampler
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset, WeightedRandomSampler
import torchvision.transforms as T


############################################################################
######                    params - do not change                      ######
############################################################################
device = "cuda" if torch.cuda.is_available() else "cpu"
if_train = False
task = 2.1

############################################################################
######                    data preprocessing - do not change          ######
############################################################################

train_leakage_loader, val_leakage_loader, test_leakage_loader, train_loader, \
      val_loader, test_loader, info_dl, train_leak_ds, train_ds, val_ds, test_ds = data_leakage_dataset(device)
n_classes = len(info_dl['label']) # liczba klas

############################################################################
######                    architecture  - do not change               ######
############################################################################
class LeakModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((7,7))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7,128), 
            nn.ReLU(),
            nn.Linear(128,2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model_leak = LeakModel().to(device)

############################################################################
######                    hiperparameters (do not change)             ######
############################################################################

# number of epochs
epochs = 40

# optimizer
opt = optim.Adam(model_leak.parameters(), lr=1e-3)

# loss function
loss_fn = nn.CrossEntropyLoss()

############################################################################
######                    training (do not change)                    ######
############################################################################
model_data_leakage = classification_training(model_leak, opt, epochs, loss_fn, device,
                                             n_classes, train_loader, val_loader,
                                             test_loader, if_train, task)

############################################################################
######                    validation (do not change)                  ######
############################################################################
print("="*70)
print(" " * 20 + "METRICS - DATA LEAKAGE")
print("="*70)
print()

classification_validate(info_dl, epochs, device, model_data_leakage, val_leakage_loader, if_train, task)

print("="*70)
print(" " * 20 + "METRICS - VALIDATION: EXTERNAL TEST DATASET")
print("="*70)
print()

classification_validate(info_dl, epochs, device, model_data_leakage, test_loader, if_train, 2.11)

print("="*70)
print(" " * 20 + "GRAD-CAM - DATA LEAKAGE")
print("="*70)
print()
binary_classification_GRADCAM(model_data_leakage, train_leak_ds, device)

print("="*70)
print(" " * 20 + "GRAD-CAM - EXTERNAL TEST DATASET")
print("="*70)
print()
binary_classification_GRADCAM(model_data_leakage, test_ds, device)

---
<h3 id="dataleakage">Questions:</h3>

`` 1. Why are the metrics values ​​on the leaked data set higher compared to the results on the external test set?``

`` 2. How can you recognize the problem of data leakage?``

`` 3. How can you avoid data leaks?``

---
Place for your response (double click to edit):

1. Because the model has access to information from the training data that should not be available during evaluation.

2. Data leakage can be suspected when the model achieves unusually high performance on validation or test data but performs significantly worse on new or external data.

3. Data leakage can be avoided by properly separating training, validation, and test datasets before any preprocessing.

---


### 2.2 Class imbalance

The data used in this task came from a database containing 1,214 images of normal lungs and 50 images of pneumonia.

``Examine the following code snippet (2.2A). Run it without making any changes. Then, complete the code in the next block (2.2B).``

In [ ]:
############################################################################
######                    CLASS IMBALANCE: 2.2A                       ######
############################################################################

from src.utils import class_imbalance_dataset
from torch.utils.data import DataLoader, Subset, ConcatDataset, WeightedRandomSampler
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset, WeightedRandomSampler
import torchvision.transforms as T


############################################################################
######                    params (do not change)                      ######
############################################################################
device = "cuda" if torch.cuda.is_available() else "cpu"
if_train = False
task = 2.2

############################################################################
######                    data preprocessing (do not change)          ######
############################################################################
rc1, train_loader, val_loader, test_loader, info_ci, train_ds, val_ds, test_ds = class_imbalance_dataset(device, True)

targets = [train_ds[i][1].item() for i in range(len(train_ds))]
class0_count = targets.count(0)
class1_count = targets.count(1)

############################################################################
######                    architecture (do not change)                ######
############################################################################
class ClassImbalanceModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((7,7))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7,128), nn.ReLU(),
            nn.Linear(128,2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model_ci = ClassImbalanceModel().to(device)

############################################################################
######                    hiperparameters (do not change)             ######
############################################################################
# Liczba epok
epochs = 20

# Optymalizator
opt = optim.Adam(model_ci.parameters(), lr=1e-3)

# Funkcja straty
loss_fn = nn.CrossEntropyLoss()

############################################################################
######                    training  (do not change)                   ######
############################################################################
print()
print()
model_data_class_imbalance = classification_training(model_ci, opt, epochs, loss_fn, device,
                                             len(info_ci['label']), train_loader, val_loader,
                                             test_loader, if_train, task)

############################################################################
######                    validation   (do not change)                ######
############################################################################

print("="*70)
print(" " * 20 + "QUALITY METRICS")
print("="*70)
print()
classification_validate(info_ci, epochs, device, model_data_class_imbalance, test_loader, if_train, task)

print("="*70)
print(" " * 20 + "GRAD-CAM")
print("="*70)
print()
binary_classification_GRADCAM(model_data_class_imbalance, test_ds, device)

<h3 id="classimbalance">Use one of the solutions that can help balance the inequality, e.g.:</h3>

- weighted loss function: [look at the *weight* parameter](https://docs.pytorch.org/docs/stable/generated/torch.nn.modules.loss.CrossEntropyLoss.html), [link #1](https://stackoverflow.com/questions/61821500/passing-weights-to-cross-entropy-loss), [link #2](https://stackoverflow.com/questions/61414065/pytorch-weight-in-cross-entropy-loss)
- weighted random sampler: [use example no. 1](https://stackoverflow.com/questions/60812032/using-weightedrandomsampler-in-pytorch) and [use example no. 2](https://www.maskaravivek.com/post/pytorch-weighted-random-sampler/)

    ``Relevant variables:``

    ``class0_count - number of class 0 images``

    ``class1_count - number of class 1 images``

    ``train_ds, val_ds, test_ds - data sets (provided as input to the DataLoader)``

In [1]:
############################################################################
######                    CLASS IMBALANCE: 2.2B                       ######
############################################################################

from src.utils import class_imbalance_dataset
from torch.utils.data import DataLoader, Subset, ConcatDataset, WeightedRandomSampler
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset, WeightedRandomSampler
import torchvision.transforms as T


############################################################################
######                    params (do not change)                      ######
############################################################################
device = "cuda" if torch.cuda.is_available() else "cpu"
if_train = True
task = 2.2

############################################################################
######                    data preprocessing (do not change)          ######
############################################################################
rc1, train_loader, val_loader, test_loader, info_ci, train_ds, val_ds, test_ds = class_imbalance_dataset(device, True)
targets = [train_ds[i][1].item() for i in range(len(train_ds))]

class0_count = targets.count(0)
class1_count = targets.count(1)

############################################################################
######                    architecture (can be changed)               ######
############################################################################
class ClassImbalanceModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((7,7))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7,128), 
            nn.ReLU(),
            nn.Linear(128,2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model_ci = ClassImbalanceModel().to(device)


############################################################################
######                    hiperparameters (CHANGE HERE!)              ######
############################################################################
# number of epochs 
epochs = 20

# optimizer 
opt = optim.Adam(model_ci.parameters(), lr=1e-3)

# Loss function
#loss_fn = nn.CrossEntropyLoss()


weights = torch.tensor([
    1.0 / class0_count,
    1.0 / class1_count
], dtype=torch.float).to(device)
weights = weights / weights.sum()

loss_fn = nn.CrossEntropyLoss(weight=weights)
############################################################################
######                    training (do not change)                    ######
############################################################################
print()
print()
model_data_class_imbalance = classification_training(model_ci, opt, epochs, 
                                                    loss_fn, device,
                                                    len(info_ci['label']), 
                                                    train_loader, val_loader,
                                                    test_loader, if_train, task)

############################################################################
######                    validation (do not change)                  ######
############################################################################

print("="*70)
print(" " * 20 + "QUALITY METRICS")
print("="*70)
print()
classification_validate(info_ci, epochs, device, model_data_class_imbalance, test_loader, if_train, task)

print("="*70)
print(" " * 20 + "GRAD-CAM")
print("="*70)
print()
binary_classification_GRADCAM(model_data_class_imbalance, test_ds, device)

ModuleNotFoundError: No module named 'pytorch_grad_cam'

### 2.3 Overfitting

For good quality, small-sized data (28x28), a deep model with a large number of parameters was prepared.

``Examine the following code snippet (2.3A). Run it without making any changes. Answer the questions, and then complete the code in the next block (2.3B).``

In [ ]:
############################################################################
######                    OVERFITTING: 2.3A                           ######
############################################################################

from src.utils import overfitting_dataset, classification_validate, classification_training, binary_classification_GRADCAM
from torch.utils.data import DataLoader, Subset, ConcatDataset, WeightedRandomSampler
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset, WeightedRandomSampler
import torchvision.transforms as T

############################################################################
######                    params (do not change)                       ######
############################################################################
if_train = False
device = "cuda" if torch.cuda.is_available() else "cpu"
task = 2.3

############################################################################
######                    data preprocessing  (do not change)         ######
############################################################################

info, train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = overfitting_dataset(device)

############################################################################
######                    architecture (do not change)                ######
############################################################################

class BigCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), 
            nn.ReLU(),
            nn.Conv2d(64,128,3,padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128,256,3,padding=1), 
            nn.ReLU(),
            nn.Conv2d(256,512,3,padding=1), 
            nn.ReLU(),
            nn.Conv2d(512,512,3,padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512*7*7, 1024), 
            nn.ReLU(),
            nn.Linear(1024, 2)
        )
    def forward(self, x):
        return self.classifier(self.features(x))
    

model_overfitting = BigCNN().to(device)

############################################################################
######                    hiperparameters  (do not change)            ######
############################################################################

# optimizer
opt = optim.Adam(model_overfitting.parameters(), lr=1e-3)
# Loss function
loss_fn = nn.CrossEntropyLoss()
# Number of epochs
epochs = 50

############################################################################
######                    training    (do not change)                 ######
############################################################################
model_data_overfitting = classification_training(model_overfitting, opt, epochs, loss_fn, device,
                                             len(info['label']), train_loader, val_loader,
                                             test_loader, if_train, task)

############################################################################
######                    validation     (do not change)              ######
############################################################################
classification_validate(info, epochs, device, model_data_overfitting, test_loader, if_train, task)
binary_classification_GRADCAM(model_data_overfitting, test_ds, device)

[<h3 id="overfitting">Questions:</h3>

`` 1. How can you recognize overfitting? What are the symptoms in this example?``

`` 2. How to prevent overfitting? ``

---

Place for your responses (double-click to edit):

1. Overfitting can be recognized when a model performs very well on the training dataset but significantly worse on the validation or test dataset.

2. Data augmentation, regularization, applying dropout layers, early stopping.

---



<h3 id="overfitting-solution">Use one of the solutions that can help with overfitting, e.g.:</h3>

- Add a dropout layer (a layer with random neuron shutdown): [more information here](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html)
- Weight decay: [for the SGD optimizer](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html#torch.optim.SGD) and [Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html#torch.optim.Adam) (look at the *weight_decay* parameter)
- Reduce the number of epochs

In [ ]:
############################################################################
######                    OVERFITTING: 2.3B                           ######
############################################################################

from src.utils import overfitting_dataset, classification_validate, classification_training, binary_classification_GRADCAM
from torch.utils.data import DataLoader, Subset, ConcatDataset, WeightedRandomSampler
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset, WeightedRandomSampler
import torchvision.transforms as T

############################################################################
######                    params (do not change)                       ######
############################################################################
if_train = True
device = "cuda" if torch.cuda.is_available() else "cpu"
task = 2.3

############################################################################
######                    data preprocessing    (do not change)       ######
############################################################################

info, train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = overfitting_dataset(device)

############################################################################
######                    architecture   (can be changed)             ######
############################################################################

class BigCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), 
            nn.ReLU(),
            nn.Conv2d(64,128,3,padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128,256,3,padding=1), 
            nn.ReLU(),
            nn.Conv2d(256,512,3,padding=1), 
            nn.ReLU(),
            nn.Conv2d(512,512,3,padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512*7*7, 1024), 
            nn.ReLU(),
            nn.Linear(1024, 2)
        )
    def forward(self, x):
        return self.classifier(self.features(x))
    

model_overfitting = BigCNN().to(device)

############################################################################
######                    hiperparameters  (can be changed)           ######
############################################################################

# optimizer
opt = optim.Adam(model_overfitting.parameters(), lr=1e-3, weight_decay=1e-4)
# loss function
loss_fn = nn.CrossEntropyLoss()
# number of epochs
epochs = 20

############################################################################
######                    training   (do not change)                  ######
############################################################################
model_data_overfitting = classification_training(model_overfitting, opt, epochs, loss_fn, device,
                                             len(info['label']), train_loader, val_loader,
                                             test_loader, if_train, task)

############################################################################
######                    validation      (do not change)             ######
############################################################################
classification_validate(info, epochs, device, model_data_overfitting, test_loader, if_train, task)
binary_classification_GRADCAM(model_data_overfitting, test_ds, device)

<h3 id="multiclass">3. Multiclass Classification</h3>

Your task is to prepare a model and select hyperparameters for multiclass classification. The database contains images of skin lesions (dermaMNIST) that belong to one of 7 classes. Try to select parameters to achieve an average accuracy of >= 0.7.

In [ ]:
############################################################################
######                    MULTICLASS CLASSIFICATION                   ######
############################################################################

from src.utils import dermaMNIST_dataset, classification_validate, classification_training, multiclass_classification_GRADCAM
from torch.utils.data import DataLoader, Subset, ConcatDataset, WeightedRandomSampler
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset, WeightedRandomSampler
import torchvision.transforms as T

############################################################################
######                    params (do not change)                       ######
############################################################################
if_train = True
device = "cuda" if torch.cuda.is_available() else "cpu"
task = 2.4

############################################################################
######                    data preprocessing   (do  not change)       ######
############################################################################

info, train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = dermaMNIST_dataset(device)

############################################################################
######                    atchitecture  (change here)                 ######
############################################################################

class YourCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4,4))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(256*4*4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(256, 7)
        )
    def forward(self, x):
        return self.classifier(self.features(x))
    

your_multiclass_model = YourCNN().to(device)

############################################################################
######                    hiperparameters     (change here)           ######
############################################################################

# optimizer 
opt = optim.Adam(your_multiclass_model.parameters(), lr=0.001, weight_decay=1e-4)###
# loss function
loss_fn = nn.CrossEntropyLoss()###
# number of epochs
epochs = 20  ###

############################################################################
######                    training (do not change)                    ######
############################################################################
model_data_multiclass= classification_training(your_multiclass_model, opt, epochs, loss_fn, device,
                                             len(info['label']), train_loader, val_loader,
                                             test_loader, if_train, task)

############################################################################
######                    validation      (do not change)             ######
############################################################################
classification_validate(info, epochs, device, model_data_multiclass, test_loader, if_train, task)
multiclass_classification_GRADCAM(info, model_data_multiclass, test_ds, device)